# Experiment

## Import libraries

In [77]:
import pandas as pd

file_path = "fdi_sector_vietstock_2000_2025.csv"

df = pd.read_csv(file_path)

# Step 1: Find the row that matches both conditions
mask = (df["Chỉ tiêu"] == "Manufacturing and Processing Industry") & (
    df["Đơn vị tính"] == "Million US Dollars"
)

# Step 2: Within that subset, find the row with the max value in Aug-2020
max_idx = (
    df.loc[mask, "Aug-2020"].str.replace(",", "", regex=False).astype(float).idxmax()
)

# Step 3: Slice from that row to the end
df = df.loc[max_idx:]

# Set indicator names as lowercase with underscores
# df["Chỉ tiêu"] = (
#     df["Chỉ tiêu"].str.lower().str.replace(",", "").str.replace(" ", "_")
# )

column_name = "Chỉ tiêu"

df[column_name] = (
    df[column_name]
    .str.lower()
    .str.replace(
        r"[^a-z0-9_\s-]", "", regex=True
    )  # remove everything except letters, numbers, underscore, space
    .str.replace(r"[\s-]+", "_", regex=True)  # replace any whitespace with underscore
)

id_vars = ["Chỉ tiêu", "Đơn vị tính"]

# Melt from wide to long format
df = df.melt(
    id_vars=id_vars,
    var_name="month_str",
    value_name="value",
)

# Clean numeric values
df["value"] = df["value"].astype(str).str.replace(",", "", regex=False)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Extract year and month
df["date"] = pd.to_datetime(df["month_str"], errors="coerce")

# Drop rows where date couldn't be parsed
df = df.dropna(subset=["date"])

# Extract numeric year, month
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year

# Use pivot_table with first() to handle duplicates
df = df.pivot_table(
    index=["year", "month"],
    columns=id_vars[0],
    values="value",
    aggfunc="first",
).reset_index()

# Sort by year and month
df = df.sort_values(["year", "month"]).reset_index(drop=True)

# Fill missing values with 0
df.fillna(0, inplace=True)

df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_22884\2712756588.py:50: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["month_str"], errors="coerce")


Chỉ tiêu,year,month,accommodation_and_food_services,administrative_and_support_service_activities,agriculture_forestry_and_fishery,arts_entertainment_and_recreation,construction,domestic_household_service_workers,education_and_training,financial_banking_and_insurance_activities,...,information_and_communication,manufacturing_and_processing_industry,mining_and_quarrying,other_service_activities,production_and_distribution_of_electricity_gas_water_and_air_conditioning,professional_scientific_and_technological_activities,real_estate_business_activities,transportation_and_warehousing,water_supply_and_waste_treatment,wholesale_and_retail_repair_of_motor_vehicles_and_motorcycles
0,2020,8,12329.93,990.33,3615.61,3391.38,13980.20,8.37,4400.41,738.97,...,3943.14,222939.9,4896.95,807.85,27574.58,3531.98,59680.52,5162.01,2859.30,8331.80
1,2020,9,12329.71,990.08,3616.51,3391.38,13975.21,8.37,4403.57,753.36,...,3946.56,222929.5,4896.95,809.35,27906.19,3577.34,59600.97,5217.30,2859.30,8326.36
2,2020,10,12475.14,990.27,3661.50,3392.66,10680.90,8.37,4403.43,753.36,...,3947.45,223619.6,4897.63,809.55,28367.76,3626.33,59878.27,5218.74,2865.42,8392.81
3,2020,11,12516.35,961.05,3664.02,3392.21,10683.43,11.07,4404.89,753.36,...,3950.95,225733.2,4897.63,844.55,28733.30,3643.84,60112.78,5235.86,2923.42,8434.23
4,2020,12,12506.70,963.38,3701.25,3391.52,10684.18,11.07,4411.27,752.76,...,3966.70,226490.2,4897.63,847.65,28921.82,3691.22,60057.32,5341.13,2923.42,8484.48
5,2021,1,12506.83,975.87,3769.90,3391.52,10675.08,11.07,4412.37,752.76,...,3970.73,228058.2,4897.63,847.65,28607.84,3686.19,60498.70,5501.77,2926.02,8519.57
6,2021,2,12518.79,975.86,3779.19,3391.52,10675.07,11.07,4412.98,752.76,...,3976.51,229081.8,4897.63,847.65,30017.27,3783.81,60748.06,5501.67,2926.02,8528.54
7,2021,3,12519.16,975.98,3779.19,3391.52,10680.17,11.07,4413.01,784.19,...,3985.53,229967.6,4897.63,847.65,33569.09,3774.90,60769.75,5501.96,2926.02,8531.22
8,2021,4,12521.70,977.10,3683.19,3391.27,10681.80,11.07,4419.71,784.19,...,4010.26,231167.9,4897.76,847.65,33561.52,3835.02,60925.45,5498.99,2897.03,8805.32
9,2021,5,12521.62,978.50,3687.23,3393.57,10685.21,11.07,4423.02,784.19,...,4026.77,232778.5,4894.76,847.65,33733.79,3845.10,61018.88,5500.45,2897.03,8835.50


In [78]:
df.columns

Index(['year', 'month', 'accommodation_and_food_services',
       'administrative_and_support_service_activities',
       'agriculture_forestry_and_fishery', 'arts_entertainment_and_recreation',
       'construction', 'domestic_household_service_workers',
       'education_and_training', 'financial_banking_and_insurance_activities',
       'healthcare_and_social_assistance_activities',
       'information_and_communication',
       'manufacturing_and_processing_industry', 'mining_and_quarrying',
       'other_service_activities',
       'production_and_distribution_of_electricity_gas_water_and_air_conditioning',
       'professional_scientific_and_technological_activities',
       'real_estate_business_activities', 'transportation_and_warehousing',
       'water_supply_and_waste_treatment',
       'wholesale_and_retail_repair_of_motor_vehicles_and_motorcycles'],
      dtype='object', name='Chỉ tiêu')